# Digital Twin Simulation: Temporal CDSS Evaluation

**Duration:** 35 minutes  
**Level:** Advanced  
**Prerequisites:** Complete [Notebook 00-05](00_quickstart.ipynb)

## Overview

This notebook introduces **Tier 1** of the BASICS-CDSS Advanced Simulation Suite: **Digital Twin Simulation**.

### What You'll Learn:

1. **Digital Twins for Healthcare**: Transform static archetypes into time-evolving patient models
2. **Physiological Disease Models**: Simulate sepsis, ARDS, and cardiac events using differential equations
3. **Temporal Perturbations**: Time-varying uncertainty (intermittent missing data, correlated noise)
4. **Counterfactual Evaluation**: "What-if" analysis for CDSS decisions
5. **Temporal Metrics**: Consistency, regret, delayed intervention risk

### Why This Matters:

Real clinical decisions unfold over **time**. A patient's condition evolves, new information arrives, and interventions take time to show effects. Digital twins enable evaluation of:

- **Temporal consistency**: Does CDSS maintain coherent recommendations?
- **Intervention timing**: What's the cost of delaying treatment?
- **Counterfactual reasoning**: Could the CDSS have made better decisions?
- **Dynamic robustness**: How does CDSS handle evolving uncertainty?

This is the **first framework** to combine digital twins with beyond-accuracy CDSS evaluation.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import BASICS-CDSS Temporal Module
from basics_cdss.temporal import (
    PatientDigitalTwin,
    DigitalTwinFactory,
    SepsisModel,
    RespiratoryDistressModel,
    CardiacEventModel,
    TemporalMaskOperator,
    TemporalNoiseOperator,
    CompositeTemporalPerturbation,
    CounterfactualEvaluator,
    MockCDSSModel,
    temporal_consistency_score,
    delayed_intervention_risk,
    counterfactual_regret,
    temporal_harm_trajectory,
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ BASICS-CDSS Temporal Module loaded successfully")

## 2. Create Your First Digital Twin

A **digital twin** is a virtual representation of a patient whose state evolves over time according to physiological models.

In [ ]:
# Define initial patient state (sepsis case)
initial_state = {
    'patient_id': 'P001',
    'age': 65,
    'temperature': 38.5,          # Fever
    'heart_rate': 110,             # Tachycardia
    'respiratory_rate': 24,        # Tachypnea
    'blood_pressure_sys': 95,      # Hypotension
    'blood_pressure_dia': 55,
    'white_blood_cell_count': 16000,  # Leukocytosis
    'lactate': 2.5,                # Elevated lactate
}

# Create digital twin with sepsis progression model
twin = PatientDigitalTwin(
    archetype_id="SEPSIS_001",
    initial_state=initial_state,
    disease_model=SepsisModel(),
    seed=42
)

print(f"Digital Twin created: {twin}")
print(f"\nInitial state:")
for key, value in twin.current_state.features.items():
    if key != 'patient_id' and key != 'age':
        print(f"  {key:30s}: {value}")

## 3. Simulate Patient Evolution (No Intervention)

Let's see what happens to this patient over 24 hours **without intervention**.

In [ ]:
# Simulate 24-hour trajectory without intervention
trajectory_no_treatment = twin.simulate(
    horizon_hours=24,
    dt=1.0,  # 1-hour time steps
    stochastic=True
)

print(f"Simulated {len(trajectory_no_treatment)} time points (t=0 to t=24h)")

# Convert to DataFrame for analysis
df_no_treatment = twin.get_trajectory_dataframe()

print(f"\nPatient state at t=24h (no treatment):")
final_state = trajectory_no_treatment[-1]
for key, value in final_state.features.items():
    if key not in ['patient_id', 'age', '_infection_severity']:
        print(f"  {key:30s}: {value:.2f}" if isinstance(value, float) else f"  {key:30s}: {value}")

print(f"\n⚠️ Infection severity increased from 0.0 to {final_state['_infection_severity']:.2f}")

### Visualize Patient Deterioration

In [ ]:
# Plot vital signs over time
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Temperature
axes[0, 0].plot(df_no_treatment['timestamp'], df_no_treatment['temperature'], 'r-', linewidth=2)
axes[0, 0].axhline(38.3, color='orange', linestyle='--', label='Fever threshold')
axes[0, 0].set_xlabel('Time (hours)')
axes[0, 0].set_ylabel('Temperature (°C)')
axes[0, 0].set_title('Temperature Over Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Heart Rate
axes[0, 1].plot(df_no_treatment['timestamp'], df_no_treatment['heart_rate'], 'b-', linewidth=2)
axes[0, 1].axhline(100, color='orange', linestyle='--', label='Tachycardia threshold')
axes[0, 1].set_xlabel('Time (hours)')
axes[0, 1].set_ylabel('Heart Rate (bpm)')
axes[0, 1].set_title('Heart Rate Over Time')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Blood Pressure
axes[0, 2].plot(df_no_treatment['timestamp'], df_no_treatment['blood_pressure_sys'], 'g-', linewidth=2)
axes[0, 2].axhline(90, color='red', linestyle='--', label='Hypotension threshold')
axes[0, 2].set_xlabel('Time (hours)')
axes[0, 2].set_ylabel('Systolic BP (mmHg)')
axes[0, 2].set_title('Blood Pressure Over Time')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Respiratory Rate
axes[1, 0].plot(df_no_treatment['timestamp'], df_no_treatment['respiratory_rate'], 'purple', linewidth=2)
axes[1, 0].axhline(20, color='orange', linestyle='--', label='Tachypnea threshold')
axes[1, 0].set_xlabel('Time (hours)')
axes[1, 0].set_ylabel('Respiratory Rate (breaths/min)')
axes[1, 0].set_title('Respiratory Rate Over Time')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# WBC
axes[1, 1].plot(df_no_treatment['timestamp'], df_no_treatment['white_blood_cell_count'], 'brown', linewidth=2)
axes[1, 1].axhline(12000, color='orange', linestyle='--', label='Leukocytosis threshold')
axes[1, 1].set_xlabel('Time (hours)')
axes[1, 1].set_ylabel('WBC (cells/μL)')
axes[1, 1].set_title('White Blood Cell Count')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Lactate
axes[1, 2].plot(df_no_treatment['timestamp'], df_no_treatment['lactate'], 'darkred', linewidth=2)
axes[1, 2].axhline(2.0, color='red', linestyle='--', label='Elevated lactate')
axes[1, 2].set_xlabel('Time (hours)')
axes[1, 2].set_ylabel('Lactate (mmol/L)')
axes[1, 2].set_title('Lactate Levels')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Patient Deterioration Without Intervention (24 hours)', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print("📉 Without treatment, the patient's condition worsens over time")

## 4. Counterfactual Analysis: What if We Intervene?

Now let's see what happens if we apply **early intervention** (antibiotics + fluids at t=0).

In [ ]:
# Clone twin for counterfactual simulation
twin_with_treatment = twin.clone()
twin_with_treatment.reset()

# Define intervention (aggressive early treatment)
intervention = {
    'antibiotic': True,
    'fluid_bolus': 1000,  # 1L IV fluids
}

# Simulate with intervention at t=0
trajectory_with_treatment = twin_with_treatment.simulate(
    horizon_hours=24,
    dt=1.0,
    intervention_schedule={0.0: intervention},
    stochastic=True
)

df_with_treatment = twin_with_treatment.get_trajectory_dataframe()

print("Intervention applied at t=0:")
print(f"  - Antibiotics: {intervention['antibiotic']}")
print(f"  - Fluid bolus: {intervention['fluid_bolus']} mL")

print(f"\nPatient state at t=24h (with treatment):")
final_treated = trajectory_with_treatment[-1]
for key in ['temperature', 'heart_rate', 'blood_pressure_sys', 'lactate', '_infection_severity']:
    if key in final_treated.features:
        value = final_treated.features[key]
        print(f"  {key:30s}: {value:.2f}" if isinstance(value, float) else f"  {key:30s}: {value}")

### Compare: Treated vs Untreated

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Temperature comparison
axes[0].plot(df_no_treatment['timestamp'], df_no_treatment['temperature'], 
             'r--', linewidth=2, label='No treatment', alpha=0.7)
axes[0].plot(df_with_treatment['timestamp'], df_with_treatment['temperature'], 
             'g-', linewidth=2, label='With treatment')
axes[0].axhline(38.3, color='gray', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Time (hours)', fontsize=12)
axes[0].set_ylabel('Temperature (°C)', fontsize=12)
axes[0].set_title('Temperature: Treatment Effect', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Blood pressure comparison
axes[1].plot(df_no_treatment['timestamp'], df_no_treatment['blood_pressure_sys'], 
             'r--', linewidth=2, label='No treatment', alpha=0.7)
axes[1].plot(df_with_treatment['timestamp'], df_with_treatment['blood_pressure_sys'], 
             'g-', linewidth=2, label='With treatment')
axes[1].axhline(90, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Time (hours)', fontsize=12)
axes[1].set_ylabel('Systolic BP (mmHg)', fontsize=12)
axes[1].set_title('Blood Pressure: Treatment Effect', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Infection severity (hidden state)
no_treat_infection = [s.features.get('_infection_severity', 0) for s in trajectory_no_treatment]
treat_infection = [s.features.get('_infection_severity', 0) for s in trajectory_with_treatment]

axes[2].plot(range(len(no_treat_infection)), no_treat_infection, 
             'r--', linewidth=2, label='No treatment', alpha=0.7)
axes[2].plot(range(len(treat_infection)), treat_infection, 
             'g-', linewidth=2, label='With treatment')
axes[2].set_xlabel('Time (hours)', fontsize=12)
axes[2].set_ylabel('Infection Severity', fontsize=12)
axes[2].set_title('Underlying Disease Progression', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Early intervention significantly improves patient outcomes")

## 5. Temporal Perturbations: Realistic Uncertainty

Real clinical data has **time-varying uncertainty**: sensors fail intermittently, measurements drift, data arrives delayed.

In [ ]:
# Create temporal perturbation operators
masker = TemporalMaskOperator(
    p_mask=0.2,  # 20% missingness rate
    feature_specific_rates={
        'lactate': 0.4,  # Lab results more likely to be missing
        'white_blood_cell_count': 0.4,
    },
    seed=42
)

noiser = TemporalNoiseOperator(
    noise_sigma=0.05,  # 5% measurement noise
    temporal_correlation=0.7,  # Correlated over time (AR(1))
    seed=43
)

# Apply perturbations
perturbed_trajectory = masker.apply(trajectory_with_treatment)
perturbed_trajectory = noiser.apply(perturbed_trajectory)

# Compute uncertainty profile
uncertainty_profile = masker.compute_uncertainty_profile(perturbed_trajectory)
print("Uncertainty Profile:")
for key, value in uncertainty_profile.items():
    print(f"  {key:15s}: {value:.2%}")

# Count missing values per time point
missing_counts = []
for state in perturbed_trajectory:
    n_masked = len(state.metadata.get('masked_features', []))
    missing_counts.append(n_masked)

print(f"\nMissing features per time point: {np.mean(missing_counts):.1f} ± {np.std(missing_counts):.1f}")

### Visualize Temporal Perturbations

In [ ]:
# Extract temperature from both trajectories
clean_temp = [s.features.get('temperature', np.nan) for s in trajectory_with_treatment]
perturbed_temp = [s.features.get('temperature', np.nan) for s in perturbed_trajectory]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperature with noise and missingness
axes[0].plot(range(len(clean_temp)), clean_temp, 'b-', linewidth=2, label='Clean data', alpha=0.7)
axes[0].plot(range(len(perturbed_temp)), perturbed_temp, 'ro', markersize=4, label='Noisy + missing', alpha=0.6)
axes[0].set_xlabel('Time (hours)', fontsize=12)
axes[0].set_ylabel('Temperature (°C)', fontsize=12)
axes[0].set_title('Temporal Perturbations: Missing Data + Noise', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Missing data heatmap
timestamps = range(len(perturbed_trajectory))
axes[1].bar(timestamps, missing_counts, color='red', alpha=0.6)
axes[1].set_xlabel('Time (hours)', fontsize=12)
axes[1].set_ylabel('Number of Missing Features', fontsize=12)
axes[1].set_title('Missingness Over Time', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("⚠️ Real-world CDSS must handle incomplete and noisy time-series data")

## 6. CDSS Evaluation with Counterfactual Reasoning

Let's evaluate different CDSS strategies using counterfactual analysis.

In [ ]:
# Create multiple digital twins
factory = DigitalTwinFactory(
    disease_model=SepsisModel(),
    seed=100
)

# Generate diverse initial states
np.random.seed(100)
initial_states = []
for i in range(20):
    state = {
        'patient_id': f'P{i:03d}',
        'age': np.random.randint(40, 80),
        'temperature': np.random.uniform(37.5, 39.5),
        'heart_rate': np.random.randint(85, 130),
        'respiratory_rate': np.random.randint(18, 30),
        'blood_pressure_sys': np.random.randint(85, 110),
        'blood_pressure_dia': np.random.randint(50, 70),
        'white_blood_cell_count': np.random.randint(12000, 20000),
        'lactate': np.random.uniform(1.5, 4.0),
    }
    initial_states.append(state)

# Create digital twins
digital_twins = []
for i, state in enumerate(initial_states):
    twin = PatientDigitalTwin(
        archetype_id=f"SEPSIS_{i:03d}",
        initial_state=state,
        disease_model=SepsisModel(),
        seed=100+i
    )
    digital_twins.append(twin)

print(f"Created {len(digital_twins)} digital twins for evaluation")

### Evaluate Three CDSS Strategies

In [ ]:
# Create CDSS models with different strategies
cdss_conservative = MockCDSSModel(strategy='conservative')
cdss_aggressive = MockCDSSModel(strategy='aggressive')
cdss_balanced = MockCDSSModel(strategy='balanced')

# Create counterfactual evaluator
evaluator = CounterfactualEvaluator(
    horizon_hours=24,
    dt=1.0,
    intervention_time=0.0
)

# Evaluate each strategy
print("Evaluating CDSS strategies (this may take a minute)...\n")

results_conservative = evaluator.evaluate(cdss_conservative, digital_twins[:5])  # Sample for speed
results_aggressive = evaluator.evaluate(cdss_aggressive, digital_twins[:5])
results_balanced = evaluator.evaluate(cdss_balanced, digital_twins[:5])

# Summarize results
summary_conservative = evaluator.summarize_results(results_conservative)
summary_aggressive = evaluator.summarize_results(results_aggressive)
summary_balanced = evaluator.summarize_results(results_balanced)

print("Strategy Comparison:")
print("=" * 70)
print(f"{'Metric':<30} {'Conservative':>12} {'Aggressive':>12} {'Balanced':>12}")
print("-" * 70)
print(f"{'Mean Harm (factual)':<30} {summary_conservative['mean_factual_harm']:>12.3f} {summary_aggressive['mean_factual_harm']:>12.3f} {summary_balanced['mean_factual_harm']:>12.3f}")
print(f"{'Mean Regret':<30} {summary_conservative['mean_regret']:>12.3f} {summary_aggressive['mean_regret']:>12.3f} {summary_balanced['mean_regret']:>12.3f}")
print(f"{'Fraction Suboptimal':<30} {summary_conservative['fraction_suboptimal']:>12.1%} {summary_aggressive['fraction_suboptimal']:>12.1%} {summary_balanced['fraction_suboptimal']:>12.1%}")
print("=" * 70)

# Interpretation
best_strategy = min(
    [('Conservative', summary_conservative['mean_factual_harm']),
     ('Aggressive', summary_aggressive['mean_factual_harm']),
     ('Balanced', summary_balanced['mean_factual_harm'])],
    key=lambda x: x[1]
)

print(f"\n✓ Best strategy: {best_strategy[0]} (mean harm = {best_strategy[1]:.3f})")
print(f"\nNote: Negative regret means CDSS was suboptimal compared to alternatives")

## 7. Temporal Metrics

Evaluate CDSS-specific temporal properties.

In [ ]:
# Simulate CDSS making repeated predictions over time
twin_test = digital_twins[0].clone()
twin_test.reset()

predictions_over_time = []
for t in range(0, 24, 2):  # Every 2 hours
    twin_test.step(dt=2.0, stochastic=False)
    prediction = cdss_balanced.predict(twin_test.current_state.features)
    predictions_over_time.append(prediction)

# Compute temporal consistency
consistency = temporal_consistency_score(
    predictions_over_time,
    window_size=3
)

print(f"Temporal Consistency Score: {consistency:.3f}")
print(f"  → 1.0 = perfectly consistent, 0.0 = constantly changing")
print(f"  → This CDSS has {'high' if consistency > 0.8 else 'moderate' if consistency > 0.5 else 'low'} consistency")

print("\nPredictions over time:")
for i, pred in enumerate(predictions_over_time):
    print(f"  t={i*2:2d}h: {pred}")

### Delayed Intervention Risk

In [ ]:
# Compare immediate vs delayed intervention
test_twin = digital_twins[0].clone()

# Immediate intervention (t=0)
test_twin.reset()
traj_immediate = test_twin.simulate(
    24, dt=1.0,
    intervention_schedule={0: {'antibiotic': True, 'fluid_bolus': 1000}},
    stochastic=False
)

# Delayed intervention (t=6h)
test_twin.reset()
traj_delayed = test_twin.simulate(
    24, dt=1.0,
    intervention_schedule={6: {'antibiotic': True, 'fluid_bolus': 1000}},
    stochastic=False
)

# Compute delay risk
delay_risk = delayed_intervention_risk(
    traj_immediate,
    traj_delayed,
    delay_hours=6
)

print("Impact of 6-Hour Delay in Intervention:")
print("=" * 50)
print(f"Cumulative harm (immediate): {delay_risk['cumulative_harm_immediate']:.2f}")
print(f"Cumulative harm (delayed):   {delay_risk['cumulative_harm_delayed']:.2f}")
print(f"Additional harm from delay:  {delay_risk['harm_increase']:.2f} (+{delay_risk['harm_increase_percent']:.1f}%)")
print(f"Harm per hour of delay:      {delay_risk['harm_per_hour_delay']:.3f}")
print("=" * 50)

if delay_risk['harm_increase'] > 0:
    print("\n⚠️ Delaying intervention increases patient harm significantly")
    print("   → CDSS should recommend early aggressive treatment for suspected sepsis")

## 8. Summary: Digital Twin Evaluation

### Key Takeaways:

1. **Digital twins enable temporal CDSS evaluation** - moving beyond static snapshots

2. **Physiological models** simulate realistic disease progression (sepsis, ARDS, cardiac events)

3. **Temporal perturbations** introduce time-varying uncertainty (missing data, noise)

4. **Counterfactual reasoning** identifies cases where CDSS could have made better decisions

5. **Temporal metrics** assess:
   - Consistency of recommendations over time
   - Cost of delayed interventions
   - Regret from suboptimal decisions

### Clinical Impact:

- **Pre-deployment testing**: Identify failure modes before real patients are affected
- **Intervention timing**: Quantify the urgency of recommended actions
- **Strategy optimization**: Compare conservative vs aggressive approaches
- **Safety assurance**: Ensure CDSS behaves coherently as patient state evolves

### Next Steps:

- **[Notebook 07](07_causal_simulation.ipynb)**: Tier 2 - Causal simulation with SCMs
- **[Notebook 08](08_multiagent_simulation.ipynb)**: Tier 3 - Multi-agent workflow simulation

---

**Novel Contribution:** This is the first framework to combine digital twin simulation with beyond-accuracy CDSS evaluation metrics for safety-critical clinical AI systems.